# SPHERE-Count — bead counter (Colab)

This notebook runs the **SPHERE-Count** area/absorbance pipeline (Method 3).

**How to run:** click **Runtime → Run all**, then upload your images when the
button appears. Or run each cell with the play button on its left.

**For each image you get**
- the bead count (report the **BEST** number)
- your image with every bead circled (red = one bead, orange = a clump)
- a QC histogram: the dashed line should sit in the empty gap between the two humps

A `results.zip` with everything downloads automatically at the end.


In [ ]:
# STEP 1 of 3 — Setup. Installs the pinned dependencies and the SPHERE-Count
# package straight from GitHub, then imports it. (~30 s.)
# EDIT the URL below to your repository once it is online:
REPO = "https://github.com/<your-username>/sphere-count.git"

!pip -q install "git+{}".format(REPO)

import io, os, csv, zipfile
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from sphere_count import count_image, overlay_image

print("Setup complete. Go to the next cell.")


In [ ]:
# STEP 2 of 3 — Upload your images. Run this cell; a "Choose Files" button
# appears below. Pick one or more bead images (JPG/PNG/TIF).
from google.colab import files
print("Click 'Choose Files' and pick your bead image(s):")
uploaded = files.upload()
print(f"\nGot {len(uploaded)} image(s):", ", ".join(uploaded.keys()))


In [ ]:
# STEP 3 of 3 — Count. Prints the count, shows beads circled and the QC
# histogram for each image, and downloads results.zip at the end.
os.makedirs("results", exist_ok=True)
rows = []
print(f"{'image':30}{'BEST':>7}{'peak':>7}{'area':>7}{'mass':>7}{'diam':>7}")
print("-" * 65)

for name, data in uploaded.items():
    img8 = np.array(Image.open(io.BytesIO(data)).convert("L")).astype(np.uint8)
    res = count_image(img8)
    stem = os.path.splitext(name)[0]
    print(f"{stem[:29]:30}{res.best:>7}{res.n_peak:>7}"
          f"{res.n_area:>7.0f}{res.n_mass:>7.0f}{res.diameter:>7.1f}")
    rows.append([stem, res.best, res.n_peak, round(res.n_area),
                 round(res.n_mass), round(res.diameter, 2)])

    ov = overlay_image(res)
    cv2.imwrite(f"results/{stem}_overlay.png", cv2.cvtColor(ov, cv2.COLOR_RGB2BGR))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    ax[0].imshow(ov)
    ax[0].set_title(f"{stem}  -  ~{res.best} beads\n(red = single, orange = clump)")
    ax[0].axis("off")
    ax[1].hist(res.cand_abs, bins=np.arange(-0.05, 1.0, 0.025),
               color="#5a7fa6", edgecolor="white")
    ax[1].axvline(0.40, color="crimson", ls="--", lw=2, label="threshold (0.40)")
    ax[1].set_yscale("log")
    ax[1].set_xlabel("relative absorbance")
    ax[1].set_title("QC: dashed line should sit in the gap\nbetween the two humps")
    ax[1].legend()
    plt.savefig(f"results/{stem}_histogram.png", dpi=130, bbox_inches="tight")
    plt.show()

with open("results/counts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "beads_best", "n_peak", "n_area", "n_mass", "bead_diameter_px"])
    w.writerows(rows)

with zipfile.ZipFile("results.zip", "w") as z:
    for root, _, fnames in os.walk("results"):
        for fn in fnames:
            z.write(os.path.join(root, fn))

print("\nBEST = the number to report.  peak = lower bound (resolvable beads only).")
print("Downloading results.zip (counts + overlays + histograms)...")
files.download("results.zip")
